# Spectral function and Green's-function determinant

Two ways of looking at the same correlated Green's function, on two closed-shell
active spaces:

- the **spectral function** $-\mathrm{Im}\,\mathrm{Tr}\,G(\omega)$, the usual density of
  states, which peaks at the **poles** of $G$;
- $\log|\det G(\omega)|$, which peaks at those same poles but *also* dips at the **zeros**
  of $\det G$.

The zeros carry no spectral weight, so the first picture is completely blind to them — yet
they are half of what determines the topology of $G$ (see
[`docs/theory.md`](../docs/theory.md)). Switching the two-electron interaction off, with
everything else held fixed, makes the point: without it, $\det G$ has poles and no zeros.

Everything here is generated from coordinates written in this notebook, and runs in a few
seconds.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pyscf import gto

from casgf import ActiveSpace, lehmann

FREQS = np.linspace(-1.5, 1.5, 1201)
ETA = 0.01

In [ ]:
def benzene(cc=1.39, ch=1.09):
    """Idealised D6h benzene: a regular hexagon of carbons with radial C-H bonds."""
    atoms = []
    for k in range(6):
        c, s = np.cos(2 * np.pi * k / 6), np.sin(2 * np.pi * k / 6)
        atoms.append(f"C {cc * c:.8f} {cc * s:.8f} 0.0")
        atoms.append(f"H {(cc + ch) * c:.8f} {(cc + ch) * s:.8f} 0.0")
    return "; ".join(atoms)


def h4(width, height=2.6):
    """Four hydrogens on the corners of a rectangle, in Bohr.

    A compact strongly correlated test case: four 1s orbitals, four electrons,
    and a real point group (D2h, rising to D4h when width == height).
    """
    x, y = width / 2, height / 2
    return "; ".join(
        f"H {sx * x:.8f} {sy * y:.8f} 0.0"
        for sx, sy in ((1, 1), (-1, 1), (-1, -1), (1, -1))
    )

In [ ]:
def with_and_without_interaction(atom, ncas, nelecas, basis, unit):
    """CASSCF, then the Green's function of the full and of the one-body Hamiltonian.

    The non-interacting case reuses the *same* CASSCF orbitals and the same h1; only
    the two-electron term is zeroed. Both are referenced to their own particle-hole
    symmetric chemical potential, so both are centred on the middle of their gap and
    the two curves can be read on one axis.
    """
    mol = gto.M(atom=atom, basis=basis, unit=unit, verbose=0)
    space = ActiveSpace.from_molecule(mol, ncas=ncas, nelecas=nelecas)
    free = ActiveSpace.from_arrays(space.h1, np.zeros_like(space.eri), space.nelecas)
    print(f"CASSCF({nelecas},{ncas})/{basis} = {space.meta['e_tot']:.8f} Ha")
    return {"interacting": lehmann(space), "non-interacting": lehmann(free)}


def plot(curves, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=120)
    for label, gf in curves.items():
        axes[0].plot(FREQS, gf.spectral(FREQS, ETA), label=label)
        axes[1].plot(FREQS, gf.log_abs_det(FREQS, ETA), label=label)
    axes[0].set_ylabel(r"$-\mathrm{Im}\,\mathrm{Tr}\,G(\omega)$")
    axes[1].set_ylabel(r"$\log|\det G(\omega)|$")
    for ax in axes:
        ax.set_xlabel(r"$\omega$ (a.u.)")
        ax.axvline(0, color="k", lw=0.5, ls="--")
        ax.legend(fontsize=9)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def summarise(curves):
    for label, gf in curves.items():
        print(f"  {label:16s} mu = {gf.mu:+.6f}   gap = {gf.gap:.6f}"
              f"   sum rule = {gf.sum_rule():.9f}")

## Benzene — CAS(6,6)/def2-SVP

The six $\pi$ orbitals.

In [ ]:
benzene_gf = with_and_without_interaction(benzene(), 6, 6, "def2-SVP", "A")
summarise(benzene_gf)
plot(benzene_gf, "Benzene, CAS(6,6)/def2-SVP")

## Rectangular H4 — CAS(4,4)/6-31G

Four hydrogens, four electrons, four orbitals: small enough to inspect element by element,
strongly correlated enough that the interaction matters.

In [ ]:
h4_gf = with_and_without_interaction(h4(2.0), 4, 4, "6-31g", "B")
summarise(h4_gf)
plot(h4_gf, "H4 rectangle, CAS(4,4)/6-31G")

## What to look at

In the right-hand panels the interacting curve dips where the non-interacting one does not.
Those dips are zeros of $\det G$, and they exist only because of the two-electron
interaction. The left-hand panels — the ordinary spectral function — show no trace of them.

The sum rule printed above is a check rather than a result: the total spectral weight has
to equal the number of active orbitals exactly, and it does, to ten digits.